# 🏥 Kaggle - Bước 1: Fine-tune LoRA (SFT Training)

Train Qwen2.5-7B-Instruct với QLoRA (4-bit) trên `bootstrap_gt/` (100 mẫu).

**Sau khi train xong:** download `lora_adapter/` rồi chuyển sang notebook `kaggle_inference.ipynb`.

**Thứ tự thực hiện:**
1. Cell 1-2: Cài đặt + kiểm tra GPU
2. Cell 3: Clone repo từ GitHub
3. Cell 4: Chọn model
4. Cell 5: Chạy training
5. Cell 6: Kiểm tra adapter đã save
6. Cell 7: Smoke test

## Cell 1 — Cài đặt thư viện

In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets
!pip install -q editdistance

## Cell 2 — Kiểm tra GPU

In [ ]:
import torch, os
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"VRAM: {gb:.1f} GB")

## Cell 3 — Clone repo từ GitHub

Thay `YOUR_GITHUB_USER` bằng username GitHub của bạn.

In [ ]:
# ===== THAY ĐỔI USER Ở ĐÂY =====
GITHUB_USER = "bphong-cyrus"   # <-- sửa lại nếu cần
REPO_NAME = "Viettel-AI-Race"
# ==============================

import os, shutil
WORK_DIR = "/kaggle/working"
REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

# Clone repo
if os.path.exists(f"{WORK_DIR}/{REPO_NAME}"):
    print(f"Repo đã tồn tại: {WORK_DIR}/{REPO_NAME}")
else:
    print(f"Cloning {REPO_URL}...")
    os.system(f"cd {WORK_DIR} && git clone {REPO_URL}")

REPO_DIR = f"{WORK_DIR}/{REPO_NAME}"
os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")
print(f"Files: {os.listdir('.')[:15]}")

## Cell 4 — Cấu hình training

Chọn base model phù hợp:

In [ ]:
# ===== CẤU HÌNH TRAINING =====

# Base model — chọn 1 trong 3:
#   "Qwen/Qwen2.5-7B-Instruct"        → tốt nhất (JSON format mạnh)
#   "Viet-Mistral/Vistral-7B-chat"    → tiếng Việt tốt nhất
#   "Qwen/Qwen2.5-3B-Instruct"        → nhanh, fit 6GB VRAM
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

# LoRA hyperparams
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training
EPOCHS = 5
LR = 1e-4
BATCH_SIZE = 1
GRAD_ACCUM = 8          # effective batch = 1*8 = 8
MAX_SEQ_LEN = 4096
WARMUP_RATIO = 0.05

# QLoRA = 4-bit (bắt buộc với T4)
USE_QLORA = True

# Output
OUTPUT_DIR = "/kaggle/working/lora_adapter"

print(f"Model: {BASE_MODEL}")
print(f"LoRA: r={LORA_R}, alpha={LORA_ALPHA}, qlora={USE_QLORA}")
print(f"Training: {EPOCHS} epochs, lr={LR}, bs={BATCH_SIZE}x{GRAD_ACCUM}")

## Cell 5 — Chạy SFT Training

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/Viettel-AI-Race')

from sft_train_lora import main as train_main
import argparse

args = argparse.Namespace(
    base_model=BASE_MODEL,
    gt_dir="bootstrap_gt",
    input_dir="input/input",
    output_dir=OUTPUT_DIR,
    epochs=EPOCHS,
    lr=LR,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    max_seq_len=MAX_SEQ_LEN,
    batch_size=BATCH_SIZE,
    grad_accum=GRAD_ACCUM,
    warmup_ratio=WARMUP_RATIO,
    qlora=USE_QLORA,
    gradient_checkpointing=True,
)

print("=" * 60)
print("BẮT ĐẦU TRAINING...")
print("=" * 60)

train_main(args)

print("=" * 60)
print("TRAINING HOÀN TẤT!")
print("=" * 60)

## Cell 6 — Kiểm tra adapter đã save

In [ ]:
import os

print(f"=== Adapter files in {OUTPUT_DIR} ===")
if os.path.exists(OUTPUT_DIR):
    total = 0
    for f in sorted(os.listdir(OUTPUT_DIR)):
        path = f"{OUTPUT_DIR}/{f}"
        sz = os.path.getsize(path) / 1024 / 1024
        total += sz
        print(f"  {f}: {sz:.2f} MB")
    print(f"\n  Tổng: {total:.1f} MB")
else:
    print("  ❌ Adapter NOT found!")

## Cell 7 — Smoke test (chạy LLM trên 1 sample)

In [ ]:
import sys, json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

sys.path.insert(0, '/kaggle/working/Viettel-AI-Race')
from llm_prompts import SYSTEM_PROMPT, build_chat_messages

# Load 4-bit base model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print("Loading base model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, OUTPUT_DIR)
model.eval()
print("✓ Model ready")

# Test on sample #1
test_text = open("input/input/1.txt", encoding="utf-8").read()[:1800]

msgs = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"Trích xuất thực thể y tế từ:\n{test_text}\n\nJSON:"},
]
prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)

print("\nGenerating (this may take ~30s)...")
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=1500,
        do_sample=False,
        repetition_penalty=1.05,
        temperature=None,
    )

gen = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n=== LLM Output ===")
print(gen[:1500])

# Try parse JSON
try:
    import re, json
    m = re.search(r'\[[\s\S]*\]', gen)
    if m:
        ents = json.loads(m.group())
        print(f"\n✓ Parsed {len(ents)} entities")
        for e in ents[:5]:
            print(f"  [{e.get('type','?')}] {e.get('text','?')}")
except Exception as ex:
    print(f"\n⚠ Parse error: {ex}")